In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import py_vollib.black_scholes.implied_volatility as iv

# Define the ticker symbol and expiration date
ticker = 'AAPL'
expiry = '2023-03-17'

# Retrieve options data from Yahoo Finance
options = yf.Ticker(ticker).option_chain(expiry)

# Extract the call and put option data
calls = options.calls
puts = options.puts

# Calculate the midpoint price of each option
calls['midpoint'] = (calls['bid'] + calls['ask']) / 2
puts['midpoint'] = (puts['bid'] + puts['ask']) / 2

# Find the at-the-money call and put options
atm_call = calls.loc[np.abs(calls['strike'] - calls['lastPrice'].mean()) == np.abs(calls['strike'] - calls['lastPrice'].mean()).min()]
atm_put = puts.loc[np.abs(puts['strike'] - puts['lastPrice'].mean()) == np.abs(puts['strike'] - puts['lastPrice'].mean()).min()]

# Calculate the implied volatility of the at-the-money options
call_iv = iv.implied_volatility(atm_call['midpoint'], atm_call['underlying_price'], atm_call['strike'], (expiry - pd.Timestamp.today()).days / 365, 0, 'c')
put_iv = iv.implied_volatility(atm_put['midpoint'], atm_put['underlying_price'], atm_put['strike'], (expiry - pd.Timestamp.today()).days / 365, 0, 'p')

# Take the average of the call and put implied volatilities
implied_vol = (call_iv + put_iv) / 2

print(f"Implied volatility of {ticker} options expiring on {expiry}: {implied_vol}")


In [ ]:
UPDATE = False
DROP_VIX = False
STATA = False
SENTIMENT = True
PresentDate = "2022-06-01"

In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections
import matplotlib.pyplot as plt
# %matplotlib inline

import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True
import datetime, pytz
from time import timezone

# Preparation of Data Frames

In [ ]:
df_sentiment=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/dfTime.pkl")

In [ ]:
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

sns.histplot(df_sentiment["compound"].loc[df_sentiment["compound"]!=0], bins=100, kde=True)


In [ ]:
import matplotlib.pyplot as plt
sns.set(font_scale=1.3)

import seaborn as sns
ax = sns.histplot(df_sentiment["compound"].loc[df_sentiment["compound"]!=0], bins=100, kde=True)

# ax = sns.distplot(df['Sentiment AR'], hist=True,color='#388E3C')
kde_x, kde_y = ax.lines[0].get_data()

x0 = -0.5
x1 =0.5
#plotting the two lines
p1 = plt.axvline(x=x0,color='#EF9A9A')
p2 = plt.axvline(x=x1,color='green')
plt.title("Minutes New York Times\nCompound Sentiment Classification\nAbove or below 0.5/-0.05")

ax.fill_between(kde_x, kde_y, where=(kde_x>x1) , 
                interpolate=True, color='#bedab1')

ax.fill_between(kde_x, kde_y, where=(kde_x<x0), 
                interpolate=True, color='#EF9A9A')

In [ ]:
df_sentiment[""]

In [ ]:
df_sentiment=pd.DataFrame()
if (UPDATE):
    df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/dfTime.pkl', compression=None)
    df_sentiment.rename(columns={"Sentiment_orig":"SentimentOrig",
                    "GI_Sentiment_orig":"GI_SentimentOrig"}
                    ,inplace=True)
    df_sentiment.index = pd.to_datetime(df_sentiment.index)
    # df_sentiment.Date = pd.to_datetime(df_sentiment.Date)
    # df_sentiment = df_sentiment[df_sentiment["NewsCount"]>0]
    # df_sentiment['Sentiment'] = df_sentiment['Sentiment'].apply(np.float)
    # df_sentiment['pos'] = df_sentiment['pos'].apply(np.float)
    # df_sentiment['neg'] = df_sentiment['neg'].apply(np.float)
    # df_sentiment['neu'] = df_sentiment['neu'].apply(np.float)
    # df_sentiment['compound'] = df_sentiment['compound'].apply(np.float)
    # df_sentiment["polarity"] = df_sentiment["polarity"].apply(np.float64)
    # df_sentiment["subjectivity"] = df_sentiment["subjectivity"].apply(np.float64)

    df_sentiment = df_sentiment.drop(df_sentiment.loc[df_sentiment['Sentiment']==0].index)
    df_sentiment.dropna(inplace=True)
    df_sentiment.to_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/df_Time.pkl', compression=None)
else:
    df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/df_Time.pkl', compression=None)

In [ ]:
df_sentiment.columns.values

In [ ]:
df_sentiment["Date"]

## VIX

In [ ]:
if (UPDATE):
    df_vix = pd.read_csv("/Users/Farid Srouji/Documents/Research/master/Data/VIX/vix-1m.csv",delimiter=";", decimal=",",names=["Date", "Minute", "Open", "High","Low","Close","Volume"])
    # tz_chicago = pytz.timezone('America/Chicago')
    df_vix.index = pd.to_datetime(df_vix['Date'] + ' ' + df_vix['Minute'], utc=True)\
                        .map(lambda x: x.tz_convert('America/Chicago'))#,dtype='datetime64[ns, Chi/Kolkata]')#tzinfo=tz_chicago)
    df_vix["Open"] = df_vix["Open"].astype(str).astype(float)
    df_vix["High"] = df_vix["High"].astype(str).astype(float)
    df_vix["Low"] = df_vix["Low"].astype(str).astype(float)
    df_vix["Close"] = df_vix["Close"].astype(str).astype(float)
    df_vix["Volume"] = df_vix["Volume"].astype(str).astype(float)

    df_vix = df_vix["2020-06-01":"2022-06-01"]
    df_vix.Date = df_vix.index
    df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/vix-1m.pkl")
else:
    df_vix = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/vix-1m.pkl")
df_vix["VIX"] = df_vix["Close"]

In [ ]:
df_vix.dropna(inplace=True)

In [ ]:
df_vix["VIX"]

# MERGING 

In [ ]:
df_sentiment = df_sentiment.reset_index().drop(columns="index").sort_values(by="Date")
df_sentiment.index = df_sentiment.Time

In [ ]:
df_sentiment["Minute"] = df_sentiment.Time.to_period(freq='T').index

In [ ]:
df_vix["Minute"] = df_vix.Date.to_period(freq='T').index

In [ ]:
# sentiment = df_sentiment.Time.dt.minute.keys().to_list()

In [ ]:
df = pd.merge(df_sentiment,df_vix, on="Minute")
df = df.sort_values(by="Minute")

In [ ]:
df.drop(columns="Date_y",inplace=True)
df.rename(columns={"Date_x":"Date"},inplace=True)

In [ ]:
df.Date

# DAY MONTH YEAR

In [ ]:
# import datetime as dt
# from datetime import date
# import calendar
# my_date = date.today()
# calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]
df['quarter'] = pd.PeriodIndex(df.Date, freq='Q')

In [ ]:
df['lead_paragraph'].apply(lambda x:len(str(x).split())).sum() #Number Of words in main text

# LN RETURN

In [ ]:
df.Date = pd.to_datetime(df.Date)
df.index = df.Date

In [ ]:
df["VIX(t-1)"] = df["VIX"].shift(periods=1)
df["Sentiment(t-1)"] = df["Sentiment"].shift(periods=1)
df["Sentiment(t-2)"] = df["Sentiment"].shift(periods=2)
df["pos(t-1)"] = df["pos"].shift(periods=1)
df["pos(t-2)"] = df["pos"].shift(periods=2)
df["neg(t-1)"] = df["neg"].shift(periods=1)
df["neg(t-2)"] = df["neg"].shift(periods=2)
df["neu(t-1)"] = df["neu"].shift(periods=1)
df["neu(t-2)"] = df["neu"].shift(periods=2)
df["compound(t-1)"] = df["compound"].shift(periods=1)
df["compound(t-2)"] = df["compound"].shift(periods=2)
df = df["2011-01-04":]

df["VIX_ln_return"]=np.log(df["VIX"]/df["VIX(t-1)"])
df["VIX_ln_return(t-1)"]=df["VIX_ln_return"].shift(periods=1)#np.log(df_sentiment["Sentiment(t-1)"]/df_sentiment["Sentiment(t-2)"])

df["Sentiment_ln_return"]=np.log(df["Sentiment"]/df["Sentiment(t-1)"])
df["Sentiment_ln_return(t-1)"]=df["Sentiment_ln_return"].shift(periods=1)#np.log(df_sentiment["Sentiment(t-1)"]/df_sentiment["Sentiment(t-2)"])

df["Sentiment_ewm10"]=df["Sentiment"].ewm(10).mean()
df["Sentiment_ewm20"]=df["Sentiment"].ewm(20).mean()

df["pos_ln_return"]=np.log(df["pos"]/df["pos(t-1)"])
df["pos_ln_return(t-1)"]=df["pos_ln_return"].shift(periods=1)#np.log(df_sentiment["pos(t-1)"]/df_sentiment["pos(t-2)"])

df["neg_ln_return"]=np.log(df["neg"]/df["neg(t-1)"])
df["neg_ln_return(t-1)"]=df["neg_ln_return"].shift(periods=1)#np.log(df_sentiment["neg(t-1)"]/df_sentiment["neg(t-2)"])

df["neu_ln_return"]=np.log(df["neu"]/df["neu(t-1)"])
df["neu_ln_return(t-1)"]=np.log(df["neu(t-1)"]/df["neu(t-2)"])

df["compound_ln_return"]=np.log(df["compound"]/df["compound(t-1)"])
df["compound_ln_return(t-1)"]=df["compound_ln_return"].shift(periods=1)#np.log(df_sentiment["compound(t-1)"]/df_sentiment["compound(t-2)"])

df["polarity(t-1)"] = df["polarity"].shift(periods=1)
df["subjectivity(t-1)"] = df["subjectivity"].shift(periods=1)

# FIXES and NULL FILL

In [ ]:
mx = df.loc[df['Sentiment_ln_return'] != np.inf, 'Sentiment_ln_return'].max()
df['Sentiment_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Sentiment_ln_return'] != -np.inf, 'Sentiment_ln_return'].min()
df['Sentiment_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['Sentiment_ln_return(t-1)'] != np.inf, 'Sentiment_ln_return(t-1)'].max()
df['Sentiment_ln_return(t-1)'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Sentiment_ln_return(t-1)'] != -np.inf, 'Sentiment_ln_return(t-1)'].min()
df['Sentiment_ln_return(t-1)'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['neg_ln_return'] != np.inf, 'neg_ln_return'].max()
df['neg_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['neg_ln_return'] != -np.inf, 'neg_ln_return'].min()
df['neg_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['neu_ln_return'] != np.inf, 'neu_ln_return'].max()
df['neu_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['neu_ln_return'] != -np.inf, 'neu_ln_return'].min()
df['neu_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['compound_ln_return'] != np.inf, 'compound_ln_return'].max()
df['compound_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['compound_ln_return'] != -np.inf, 'compound_ln_return'].min()
df['compound_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['pos_ln_return'] != np.inf, 'pos_ln_return'].max()
df['pos_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['pos_ln_return'] != -np.inf, 'pos_ln_return'].min()
df['pos_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['neg_ln_return'] != np.inf, 'neg_ln_return'].max()
df['neg_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['neg_ln_return'] != -np.inf, 'neg_ln_return'].min()
df['neg_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['neu_ln_return'] != np.inf, 'neu_ln_return'].max()
df['neu_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['neu_ln_return'] != -np.inf, 'neu_ln_return'].min()
df['neu_ln_return'].replace(-np.inf,mn,inplace=True)

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]
df['quarter'] = pd.PeriodIndex(df.Date, freq='Q')

In [ ]:
df.rename(columns={"vix":"VIX",
                "vix(t-1)":"VIX(t-1)",
                "VIX_ln_return":"VIX return",
                "sp500":"S&P500",
                "sp500(t-1)":"S&P500(t-1)",
                "sp500(t-2)":"S&P500(t-2)",
                "sp500_ln_return":"S&P500 return",
                "Sentiment_ln_return":"Sentiment return",
                "Sentiment_ln_return(t-1)":"Sentiment return(t-1)",
                "SentimentOrig_ln_return":"SentimentOrig return",
                "SentimentOrig_ln_return(t-1)":"SentimentOrig return(t-1)",
                "pos":"Positive",
                "pos(t-1)":"Positive(t-1)",
                "pos(t-2)":"Positive(t-2)",
                "pos_ln_return":"Positive return",
                "pos_ln_return(t-1)":"Positive return(t-1)",
                "neg":"Negative",
                "neg(t-1)":"Negative(t-1)", 
                "neg(t-2)":"Negative(t-2)",
                "neg_ln_return":"Negative return", 
                "neg_ln_return(t-1)":"Negative return(t-1)", 
                "neu":"Neutral", 
                "neu(t-1)":"Neutral(t-1)",
                "neu(t-2)":"Neutral(t-2)",
                "neu_ln_return":"Neutral return",
                "neu_ln_return(t-1)":"Neutral return(t-1)",
                "compound":"Compound",
                "compound(t-1)":"Compound(t-1)",
                "compound(t-2)":"Compound(t-2)",
                "compound_ln_return":"Compound return",
                "compound_ln_return(t-1)":"Compound return(t-1)",
                "polarity":"Polarity",
                "polarity(t-1)":"Polarity(t-1)",
                "polarityB":"PolarityB",
                "subjectivity":"Subjectivity",
                "subjectivity(t-1)":"Subjectivity(t-1)",
                "NewsCount_ln_return":"NewsCount return",
                "NewsCount_ln_return(t-1)":"NewsCount return(t-1)",
                "WordCount_ln_return":"WordCount return",
                "WordCount_ln_return(t-1)":"WordCount return(t-1)",
                "day":"Day of Week",
                "month":"Month",
                "month-year":"Month-Year",
                "quarter":"Quarter",
                "year":"Year",
                "GI_Sentiment_ln_return":"GI_Sentiment return",
                "GI_Sentiment_ln_return(t-1)":"GI_Sentiment return(t-1)",
                "GI_SentimentOrig_ln_return":"GI_SentimentOrig return",
                "GI_SentimentOrig_ln_return(t-1)":"GI_SentimentOrig return(t-1)",
                "GI_Negative_ln_return":"GI_Negative return",
                "GI_Negative_ln_return(t-1)":"GI_Negative return(t-1)",
                "GI_Positive_ln_return":"GI_Positive return",
                "GI_Positive_ln_return(t-1)":"GI_Positive return(t-1)"}
                ,inplace=True)

# STATA

In [ ]:
if (STATA):
    # df = df.round(2)
    # df.dropna(inplace=True)
    df.rename(columns={"Sentiment return":"sentiment_ln_return",
                "Sentiment return(t-1)":"sentiment_ln_return_lag1",
                "Positive":"pos",
                "Positive(t-1)":"pos(t-1)",
                "Positive(t-2)":"pos(t-2)",
                "Positive return":"pos_ln_return",
                "Positive return(t-1)":"pos_ln_return(t-1)",
                "Negative":"neg",
                "Negative(t-1)":"neg(t-1)", 
                "Negative(t-2)":"neg(t-2)",
                "Negative return":"neg_ln_return", 
                "Negative return(t-1)":"neg_ln_return(t-1)", 
                "Neutral":"neu", 
                "Neutral(t-1)":"neu(t-1)",
                "Neutral(t-2)":"neu(t-2)",
                "Neutral return":"neu_ln_return",
                "Neutral return(t-1)":"neu_ln_return(t-1)",
                "Compound":"compound",
                "Compound(t-1)":"compound(t-1)",
                "Compound(t-2)":"compound(t-2)",
                "Compound return":"compound_ln_return",
                "Compound return(t-1)":"compound_ln_return(t-1)",
                "Polarity":"polarity",
                "Polarity(t-1)":"polarity(t-1)",
                "PolarityB":"polarityB",
                "Subjectivity":"subjectivity",
                "Subjectivity(t-1)":"subjectivity(t-1)",
                "NewsCount return":"NewsCount_ln_return",
                "NewsCount return(t-1)":"NewsCount_ln_return(t-1)",
                "WordCount return":"WordCount_ln_return",
                "WordCount return(t-1)":"WordCount_ln_return(t-1)",
                "Day of Week":"day",
                "Month":"month",
                "Month-Year":"month-year",
                "Quarter":"quarter",
                "Year":"year",
                "GI_Sentiment return":"GI_Sentiment_ln_return",
                "GI_Sentiment return":"GI_Sentiment_ln_return",
                "GI_Negative return":"GI_Negative_ln_return",
                "GI_Positive return":"GI_Positive_ln_return"}
                ,inplace=True)
    df.to_csv("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/stata/Sentiment.csv")
    print("STATA FILE CREATED")

In [ ]:
df.columns

In [ ]:
df["WordCount"] = df["lead_paragraph"].apply(lambda x:len(str(x).split()))
df["lead_paragraph"].apply(lambda x:len(str(x).split())).sum()

In [ ]:
df.index = pd.to_datetime(df.Time)
df.index = df.Time.to_period(freq='T').index

In [ ]:
df['Time']=df['Time'].dt.strftime('%Y-%m-%d %H')

In [ ]:
df["VIX return"]

In [ ]:
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/dfTimeComplate.pkl")